# OLCF Resource Discovery & Job Submission via the AmSC Python Client

This notebook demonstrates how to query OLCF resources (including open and moderate enclaves), how to obtain an S3M token, and submit and manage compute jobs on OLCF resources (ACE Testbed, Odo, Frontier, etc.) using the AmSC Python Client's facility integration.

**What you'll do:**

1. Create an AmSC client
2. Connect to the OLCF facility and explore available compute resources
3. Obtain a S3M Token 
4. Submit a "hello world" job to Defiant or Frontier
5. Monitor the job until completion

**Prerequisites:**

- `amsc-client` installed (with globus-sdk)
- A valid OLCF account and project allocation (visit link for more information: )
- A valid S3M Token

**Authentication:**

- OLCF facility endpoints (resource listing) are public — no login needed
- Compute job submission requires an S3M token

**Coming Soon**
- OLCF Incidents
- Filesystem API
- Integration with AmSC Token

In [1]:
import os
import time
from pathlib import Path
import amsc_client
print(amsc_client.version_info())
from amsc_client import Client, Resource, Job, ApiError

{'amsc_client': '0.4.3', 'amsc_api_autogen': '0.4', 'iri_api_autogen': '0.1.2', 'git_commit': '764cdb1', 'python': '3.13.5'}


## Step 1: Create the AmSC Client

The client manages authentication and provides access to all AmSC services. For this tutorial, we only need facility access — no catalog auth is required.

In [2]:
# Create a minimal client — we only need facility access, not catalog
client = Client(token="not-needed-for-facilities")
print("✅ Client created")

# Included catalog config in case you would like to expand and play with this notebook:
# base_url='https://api.american-science-cloud.org/api/current'
# globus_app_id='e4f48665-38b5-4833-a89e-849c71f5b3e3' # ChildersAmSC ->  AmSC CLI Client
# client = Client(
#     base_url=base_url,
#     auth_method="globus",
#     globus_client_id=globus_app_id,
#     requested_scopes=f'openid profile email https://auth.globus.org/scopes/{globus_app_id}/amsc_test',
#     resource_server=globus_app_id,
#     use_id_token=True,
# )

✅ Client created


## Step 2: Connect to OLCF and Explore Resources

OLCF is not a built-in facility so you will need to manually register and configure based on the compute enclave of interest.

In [3]:
# To use S3M at OLCF you will need an S3M token (visit myOLCF to generate a token for your project).
# Uncomment the following 3 lines if have a S3M token and modify the Path.
s3m_token = Path("/Users/3ue/S3M/.env").read_text().strip() # Not needed for status -- will need later in tutorial.
#if not s3m_token:
#    raise ValueError("S3M token file is empty")

client = Client(token="not-needed-for-facilities")

In [5]:
# Connect to OLCF enclaves via manual facility registration.
# The dictionary keys are client facility names. Python variables use underscores.
OLCF_FACILITY_CONFIGS = {
    "olcf-open": {
        "base_url": "https://amsc-open.s3m.olcf.ornl.gov/",
        "display_name": "Oak Ridge Leadership Computing Facility (Open)",
    },
    "olcf-moderate": {
        "base_url": "https://amsc-moderate.s3m.olcf.ornl.gov/", 
        "display_name": "Oak Ridge Leadership Computing Facility (Moderate)",
    },
}

olcf_facilities = {}
for facility_name, facility_config in OLCF_FACILITY_CONFIGS.items():
    client.register_facility(
        facility_name,
        base_url=facility_config["base_url"],
        display_name=facility_config["display_name"],
        auth_method="token",
        token=s3m_token,
    )
    olcf_facilities[facility_name] = client.facility(facility_name)

olcf_open = olcf_facilities["olcf-open"]
olcf_moderate = olcf_facilities["olcf-moderate"]

# Downstream cells use `olcf`; switch this value to choose the active enclave.
ACTIVE_OLCF_FACILITY = "olcf-moderate" # Change this is want open vs moderate
olcf = olcf_facilities[ACTIVE_OLCF_FACILITY]
print(f"Using {ACTIVE_OLCF_FACILITY}: {olcf.base_url}")

Using olcf-moderate: https://amsc-moderate.s3m.olcf.ornl.gov


In [6]:
# List all available resources
resources = olcf.resources()

print(f"Available Resources ({len(resources)}):\n")
print(f"{'Name':12s}  {'Type':10s}  {'Status'}")
print(f"{'─'*12}  {'─'*10}  {'─'*12}")
for r in resources:
    print(f"{r.name:12s}  {r.resource_type:10s}  {r.status}")

Available Resources (1):

Name          Type        Status
────────────  ──────────  ────────────
Frontier      compute     up


In [7]:
# Modify for a specific compute resource by name (case-insensitive) -- must be in correct olcf enclave
defiant = olcf.resource("Frontier")

print(f"Resource: {defiant.name}")
print(f"ID:       {defiant.id}")
print(f"Type:     {defiant.resource_type}")
print(f"Status:   {defiant.status}")

Resource: Frontier
ID:       5173cdb4-d82f-5c81-8ef0-598997c12813
Type:     compute
Status:   up


## Step 3: Check for Active Incidents 

**This is unavailable at this time, will return a 404 error**

Before submitting a job, it's good practice to check for any active outages or maintenance windows.

In [8]:
incidents = olcf.incidents()
print(f"Total incidents: {len(incidents)}")

# Show the 5 most recent
if incidents:
    print(f"\nRecent incidents:")
    for inc in incidents[:5]:
        print( f' - {str(inc.last_modified):20} | {inc.name:25} -> {inc.resolution:20} ')

Total incidents: 1

Recent incidents:
 - 2026-09-16 19:21:08+00:00 | File system hardware maintenance -> completed            


# Step 4: Configure and Submit a Job

Now we'll submit a simple "hello world" job to on OLCF compute resource. Depending on which enclave you are authorized to use this could be Defiant, Wombat, or Quokka (ACE Testbed), Odo, or Frontier.

⚠️ You need to authenticate with your OLCF account to submit jobs. A valid S3M token for the OLCF account and project is required and must be scoped for the specific computing enclave.

**Before running: Update the configuration below, providing the corresponding account, username, queue, and directory to match your OLCF project allocation and provide a current S3M Token in step 2.** (To learn about S3M Token management see: https://docs.olcf.ornl.gov/services_and_applications/s3m/overview.html#token-management)

Queue Options:
- Frontier: batch or extended
- Defiant: batch-gpu, batch-cpu, or cron
- Quokka:
- Wombat: gh, Ampere-gpu, Ampere, or A64fx
- Odo: batch or cron

In [10]:
# ── Job Configuration ──────────────────────────────────────────────
# Update these to match your OLCF account:

OLCF_ACCOUNT  = "stf053"                      # Modify this to your OLCF project allocation
OLCF_USERNAME = "etzbd"                          # Modify this to your OLCF username
OLCF_QUEUE    = "batch"                           # Modify this to queue of interest (common error is using wrong/mismatched queue)
OUTPUT_DIR    = f"/lustre/orion/stf053/proj-shared/amsc-iri/"  # Modify, This must exist on the filesystem and have access from project auser.

# NOTE: The IRI API submits jobs using a service account or automation user (prj###_auser). The OUT_DIR above must have the correct permissions for the prj###_auser otherwise an error will occur. 
# NOTE: may need to change permission using "chmod g+rwx /lustre/polis/prj###/proj-shared/working-dir/"

# ─────────────────────────────────────────────────────────────────

import datetime
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
JOB_NAME = f"amsc-tutorial-{RUN_ID}"

print(f"Job name:    {JOB_NAME}")
print(f"Resource:    {defiant.name}")
print(f"Queue:       {OLCF_QUEUE}")
print(f"Account:     {OLCF_ACCOUNT}")
print(f"Output dir:  {OUTPUT_DIR}")

Job name:    amsc-tutorial-20260922-114344
Resource:    Frontier
Queue:       batch
Account:     stf053
Output dir:  /lustre/orion/stf053/proj-shared/amsc-iri/


In [11]:
# Submit a simple diagnostic job that runs for at least one minute.

JOB_NODE_COUNT = 1
JOB_RUNTIME_SECONDS = 30

job_script = f"""
set -e

echo "Hello from AmSC Python Client!"
echo "Run ID: {RUN_ID}"
echo "Started: $(date -Is)"
echo "Hostname: $(hostname)"
echo "User: $(whoami 2>/dev/null || id -un)"
echo "Working directory: $(pwd)"
echo "Requested nodes: {JOB_NODE_COUNT}"
echo

echo "Scheduler environment:"
for key in SLURM_JOB_ID SLURM_JOB_NAME SLURM_NNODES SLURM_JOB_NODELIST; do
    eval "value=\\${{$key:-}}"
    if [ -n "$value" ]; then
        echo "  $key=$value"
    fi
done
echo

echo "Allocated nodes:"
if [ -n "${{SLURM_JOB_NODELIST:-}}" ] && command -v scontrol >/dev/null 2>&1; then
    scontrol show hostnames "$SLURM_JOB_NODELIST"
else
    hostname
fi
echo

echo "Sleeping for {JOB_RUNTIME_SECONDS} seconds so the job remains visible in the queue."
sleep {JOB_RUNTIME_SECONDS}
echo "Finished: $(date -Is)"
""".strip()

job = defiant.submit(
    executable="/bin/bash",
    arguments=["-lc", job_script],
    directory=OUTPUT_DIR,
    name=JOB_NAME,
    queue=OLCF_QUEUE,
    account=OLCF_ACCOUNT,
    duration=300,             # Wall time in seconds (5 minutes)
    nodes=JOB_NODE_COUNT,
    environment={
        "AMSC_RUN_ID": RUN_ID,
        "AMSC_SLEEP_SECONDS": str(JOB_RUNTIME_SECONDS),
    },
)

print(f"\n✅ Job submitted!")
print(f"   Job ID:    {job.id}")
print(f"   State:     {job.state}")
print(f"   {job!r}")


✅ Job submitted!
   Job ID:    5527885
   State:     queued
   Job(id='5527885', state='queued')


## Step 5: Monitor the Job

The `Job` object is "self-aware" — it knows which facility and resource it belongs to,
and can refresh its own status from the API.

We can poll manually or use `job.wait()` to block until completion.

In [12]:
# Manual polling — check status every 5 seconds

POLL_TIMEOUT = 60  # seconds
POLL_INTERVAL = 5  # seconds

print(f"Polling job {job.id} (up to {POLL_TIMEOUT}s)...\n")
start_time = time.time()

while time.time() - start_time < POLL_TIMEOUT:
    elapsed = int(time.time() - start_time)
    try:
        current_status = job.status  # calls the API
        state = job.state
        print(f"  [{elapsed:3d}s] State: {state}")

        if job.is_terminal:
            exit_code = job.exit_code
            message = job.message
            print(f"\n✅ Job finished!")
            print(f"   Final state: {state}")
            if exit_code is not None:
                print(f"   Exit code:   {exit_code}")
            if message:
                print(f"   Message:     {message}")
            break
    except Exception as e:
        # Completed Slurm jobs disappear from the queue;
        # the API returns 400 "not found" in that case
        if "not found" in str(e).lower() or "400" in str(e):
            print(f"  [{elapsed:3d}s] Job no longer in scheduler queue (likely completed)")
            print(f"\n✅ Job completed (exited scheduler)")
            break
        else:
            print(f"  [{elapsed:3d}s] Status check error: {e}")

    time.sleep(POLL_INTERVAL)
else:
    print(f"\n⏰ Timed out after {POLL_TIMEOUT}s (job may still be running)")
    print(f"   Last known state: {job.state}")

Polling job 5527885 (up to 60s)...

  [  0s] State: active
  [  5s] State: active
  [ 10s] State: active
  [ 15s] State: active
  [ 20s] State: active
  [ 25s] State: active
  [ 30s] State: completed

✅ Job finished!
   Final state: completed
   Exit code:   0
   Message:     None


## Step 6: Read Job Output via the Filesystem API

**NOTE: The Filesystem API is not available yet to use with OLCF.**

Now that the job has completed, we can read its output directly
using the **filesystem API** — no need to SSH into the machine.


## Alternative: Using `job.wait()`

Instead of manual polling, you can use the built-in `wait()` method
which blocks until the job reaches a terminal state:

In [13]:
JOB_NODE_COUNT = 1
JOB_RUNTIME_SECONDS = 60

job_script = f"""
set -e

echo "Hello from AmSC Python Client!"
echo "Run ID: {RUN_ID}"
echo "Started: $(date -Is)"
echo "Hostname: $(hostname)"
echo "User: $(whoami 2>/dev/null || id -un)"
echo "Working directory: $(pwd)"
echo "Requested nodes: {JOB_NODE_COUNT}"
echo

echo "Scheduler environment:"
for key in SLURM_JOB_ID SLURM_JOB_NAME SLURM_NNODES SLURM_JOB_NODELIST; do
    eval "value=\\${{$key:-}}"
    if [ -n "$value" ]; then
        echo "  $key=$value"
    fi
done
echo

echo "Allocated nodes:"
if [ -n "${{SLURM_JOB_NODELIST:-}}" ] && command -v scontrol >/dev/null 2>&1; then
    scontrol show hostnames "$SLURM_JOB_NODELIST"
else
    hostname
fi
echo

echo "Sleeping for {JOB_RUNTIME_SECONDS} seconds so the job remains visible in the queue."
sleep {JOB_RUNTIME_SECONDS}
echo "Finished: $(date -Is)"
""".strip()

job2 = defiant.submit(
    executable="/bin/bash",
    arguments=["-lc", job_script],
    directory=OUTPUT_DIR,
    name=JOB_NAME,
    queue=OLCF_QUEUE,
    account=OLCF_ACCOUNT,
    duration=300,             # Wall time in seconds (5 minutes)
    nodes=JOB_NODE_COUNT,
    environment={
        "AMSC_RUN_ID": RUN_ID,
        "AMSC_SLEEP_SECONDS": str(JOB_RUNTIME_SECONDS),
    },
)

print(f"\n✅ Job submitted!")
print(f"   Job ID:    {job.id}")
print(f"   State:     {job.state}")
print(f"   {job!r}")

try:
    job2.wait(timeout=120, poll_interval=5)
    print(f"✅ Job completed: state={job2.state}, exit_code={job2.exit_code}")
except TimeoutError:
    print(f"⏳ Job still running after timeout (state: {job2.state})")


✅ Job submitted!
   Job ID:    5527885
   State:     completed
   Job(id='5527885', state='completed')
✅ Job completed: state=completed, exit_code=0


## Summary

This tutorial showed how to:

| Step | API Call | Auth Required? |
|------|---------|----------------|
| Connect to OLCF | `client.facility("olcf")` | No |
| List resources | `olcf.resources()` | No |
| Get resource details | `olcf.resource("Defiant")` | No |
| Submit a job | `defiant.submit(...)` | Yes (S3M Token) |
| Check job status | `job.status` | Yes (S3M Token) |
| Wait for completion | `job.wait(timeout=120)` | Yes (S3M Token) |

**Key concepts:**
- **Resource-scoped compute**: Jobs are submitted via the resource object (`defiant.submit(...)`) rather than a separate compute client
- **Self-aware jobs**: The `Job` object tracks its own facility and resource, and can refresh status, wait, or cancel itself
- **Filesystem operations are currently unavailable at OLCF**
- **Incidents is currently not supported at OLCF**
- **Jobs submit as project automation user (prj###_auser)**: Need to make sure working directory has correct permisions

## Troubleshooting

OLCF is actively trying to improve the IRI/S3M error tracking and messaging. Please contact OLCF Help (help@olcf.ornl.gov) to provide feedback and request assistance. 
